In [0]:
%sql
CREATE TABLE IF NOT EXISTS trainings (
  id INT,
  training_name STRING,
  duration_hours INT,
  skill_learned STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO trainings (id, training_name, duration_hours, skill_learned)
VALUES 
  (1, 'Databricks SQL Fundamentals', 10, 'SQL & Lakehouse'),
  (2, 'Delta Lake Deep Dive', 15, 'ACID Transactions'),
  (3, 'Unity Catalog Administration', 20, 'Data Governance'),
  (4, 'PySpark Optimization', 25, 'Performance Tuning'),
  (5, 'Data Visualization with Dashboards', 12, 'BI Reporting');

In [0]:
%sql
-- 1. Insert a new record (Creates Version 1)
INSERT INTO trainings (id, training_name, duration_hours, skill_learned)
VALUES (6, 'Data Engineering Masterclass', 40, 'Advanced Spark');

In [0]:
%sql
-- 2. Update a record (Creates Version 2)
UPDATE trainings 
SET duration_hours = 12 
WHERE training_name = 'Databricks SQL Fundamentals';
-- 3. Delete a record (Creates Version 3)
DELETE FROM trainings 
WHERE id = 5;

In [0]:
%sql DESCRIBE HISTORY trainings;

In [0]:
%sql select * from trainings

In [0]:
%sql
SELECT * FROM trainings VERSION AS OF 2;


In [0]:
%sql SELECT * FROM trainings TIMESTAMP AS OF '2026-03-30T06:26:45.000+00:00';

In [0]:
%sql RESTORE TABLE trainings TO VERSION AS OF 4;

In [0]:
%sql
CREATE OR REPLACE TABLE traings_shallow_clone SHALLOW CLONE trainings VERSION AS OF 2;

In [0]:
%sql
ALTER TABLE trainings SET TBLPROPERTIES (
  'delta.logRetentionDuration' = 'interval 10 days',
  'delta.deletedFileRetentionDuration' = 'interval 10 days'
);

In [0]:
%sql
CREATE OR REPLACE VIEW trainings_summary_view AS
SELECT 
  training_name, 
  skill_learned 
FROM trainings 
WHERE duration_hours > 10;

In [0]:
%sql
SELECT * FROM system.information_schema.views 
WHERE table_name = 'trainings_summary_view';

In [0]:
%sql
CREATE MATERIALIZED VIEW trainings_by_skill_mv
AS 
SELECT 
  skill_learned, 
  COUNT(id) AS total_trainings, 
  SUM(duration_hours) AS total_duration
FROM trainings
GROUP BY skill_learned;


In [0]:
%sql
ALTER TABLE trainings CLUSTER BY (training_name);

In [0]:
%sql
OPTIMIZE trainings;

In [0]:
%sql
-- Remove Liquid clustering to use ZORDER BY
ALTER TABLE trainings CLUSTER BY NONE;

-- Now apply ZORDER BY optimization
OPTIMIZE trainings ZORDER BY (id);

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define Schema and Data for df1
schema1 = StructType([
    StructField("empno", IntegerType(), True),
    StructField("ename", StringType(), True),
    StructField("sal", IntegerType(), True)
])
data1 = [(1, "peter", 1000), (2, "sam", 3000), (4, "arun", 5000), (6, "sai", 2300)]
df1 = spark.createDataFrame(data1, schema1)

# Define Schema and Data for df2
schema2 = StructType([
    StructField("empno", IntegerType(), True),
    StructField("loc", StringType(), True)
])
data2 = [(1, "hyd"), (2, "del"), (3, "chn"), (5, "pun")]
df2 = spark.createDataFrame(data2, schema2)

# Save as Delta Tables
df1.write.format("delta").mode("overwrite").saveAsTable("emp_details")
df2.write.format("delta").mode("overwrite").saveAsTable("emp_locations")


In [0]:
%sql
SELECT 
  COALESCE(a.empno, b.empno) AS empno,
  a.ename,
  a.sal,
  b.loc
FROM emp_details a
FULL OUTER JOIN emp_locations b
  ON a.empno = b.empno;

  